# ДЗ1. Классификация текстов

В этой работе мы попробуем построить **спам-фильтр** для музыкальных видео. Обычно в комментариях к музыкальному видео вперемешку живут обсуждение песни, реклама чужих каналов, ссылки и просьбы подписаться. Мы попробуем сделать фильтр, который можно переносить на новое видео.

Исходный классификатор уже есть. Ваша задача сейчас - выяснить, какую информацию ему стоит оставить, придумать собственные признаки и проверить, как он реагирует на изменения в тексте.

Работа оценивается в **10 баллов**, распределение баллов отражено в таблице:

| Задание | Баллы |
|---|---:|
| 1. Подготовка данных и разметка | 2 |
| 2. Обработка под задачу | 2 |
| 3. Собственные признаки | 2 |
| 4. Что дает каждый набор признаков | 1.5 |
| 5. Проверки на изменённых текстах | 1.5 |
| 6. Новое видео | 1 |

По ходу оставляйте короткие ответы на вопросы. Корректный эксперимент, который не улучшил качество, оценивается так же, как эксперимент с улучшением. Фиксированного порога метрики для получения баллов нет.

**Формат сдачи**

Задание сдается через систему LMS. Посылка будет содержать:

- ноутбук `homework-01-Username.ipynb`, где Username - ваша фамилия латиницей без пробелов (например, `homework-01-Ivanov.ipynb`);
- файл `test_predictions.csv`.

Все таблицы, графики и выводы оставьте в тетрадке. Перед сдачей перезапустите ядро и выполните ячейки по порядку, чтобы проверить воспроизводимость результата.

> Источники и помощь генеративных моделей укажите в конце, но предполагается, что ключевые решения нужно уметь объяснить, поэтому размышления строго рекомендуется формулировать без LLM. По усмотрению ассистента при изобилии сгенерированного текста и кода балл может быть снижен.

## Подготовка

В работе мы будем использовать [YouTube Spam Collection](https://archive.ics.uci.edu/dataset/380/youtube+spam+collection) - это реальные комментарии к пяти музыкальным видео. Метка **1** обозначает спам, **0** - обычный комментарий. Это небольшой корпус 2015 года, на нём как раз будет удобно проверять идеи.

В исходном варианте 1956 комментариев.

Файлы лежат рядом с тетрадкой в `data/`. В Colab можно использовать ссылки из следующей ячейки или загрузить эту папку рядом с ноутбуком.

In [2]:
%pip install -q "pandas==2.2.3" "scikit-learn==1.6.1" "matplotlib==3.9.4"

In [3]:
from pathlib import Path
import html
import re
import platform
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, ConfusionMatrixDisplay

SEED = 13
pd.set_option('display.max_colwidth', 200)
plt.rcParams.update({'figure.figsize': (8, 4), 'font.size': 11})

In [5]:
DATA_URL = (
    'https://raw.githubusercontent.com/AlbinaBurlova/hse-bda-nlp-course/'
    '2026/homeworks/HW1_text_classification/data/'
)
data_path = Path('data')

files = {
    'psy': 'Youtube01-Psy.csv',
    'katyperry': 'Youtube02-KatyPerry.csv',
    'lmfao': 'Youtube03-LMFAO.csv',
    'eminem': 'Youtube04-Eminem.csv',
    'shakira': 'Youtube05-Shakira.csv',
}
paths = {
    source: data_path / filename if (data_path / filename).exists() else DATA_URL + filename
    for source, filename in files.items()
}

**О задаче**

В исходных файлах `CONTENT` - текст, `CLASS` - метка, `COMMENT_ID` - идентификатор комментария, `AUTHOR` - имя автора, `DATE` - дата. Видео определяется файлом.

Мы проверяем перенос **на другое видео**. Разбиение фиксируем: трейн - Psy, Katy Perry и LMFAO; валидация - Eminem; тест - Shakira.

Схему, пропуски и повторы можно проверять во всей выгрузке. Идеи обработки и признаков берём из тренировочной выборки, модели сравниваем на валидации, тест используем для итоговой оценки. Модель получает только текст, метаданные не входят в признаки.

Основная метрика - **F1 спама, класса 1**. Рядом желательно показать precision и recall, ведь ошибочно скрытый обычный комментарий и пропущенная реклама - разные ошибки.

## 1. Подготовка данных и разметка [2 балла]

### 1.1. От файлов к таблице

Загрузите пять CSV через `pd.read_csv` и объедините в порядке из `files`. Добавьте `source` и присвойте строкам `id` от 0 до удаления каких-либо записей.

Переименуйте `CONTENT` в `text`, `CLASS` в `label` - это чтобы выдерживать нужный нейминг. Исходный текст тоже сохраните.

Проверьте состав данных по источникам и классам, пропуски и пустые тексты. Покажите, какие данные потеряются при удалении всех строк с пропусками. Оправданы ли эти потери для нашей задачи?

In [6]:
df = pd.concat(
    [
        pd.read_csv(paths[s]).assign(source=s)
        for s in files
        ],
    ignore_index=True
)
df["id"] = df.index
df = df.rename(columns={"CONTENT": "text", "CLASS": "label"})
df["text_raw"] = df["text"]

display(pd.crosstab(df.source, df.label).reindex(files))
display(df.isna().sum())

empty = df.text.str.strip().eq("")
display(df.loc[empty])

lost = df[df.isna().any(axis=1)]
display(lost)
display(pd.crosstab(lost.source, lost.label))

label,0,1
source,,
psy,175,175
katyperry,175,175
lmfao,202,236
eminem,203,245
shakira,196,174


,0
COMMENT_ID,0
AUTHOR,0
DATE,245
text,0
label,0
source,0
id,0
text_raw,0


,COMMENT_ID,AUTHOR,DATE,text,label,source,id,text_raw


,COMMENT_ID,AUTHOR,DATE,text,label,source,id,text_raw
1138,z12rwfnyyrbsefonb232i5ehdxzkjzjs2,Lisa Wellas,NaN,+447935454150 lovely girl talk to me xxx﻿,1,eminem,1138,+447935454150 lovely girl talk to me xxx﻿
1140,z13vsfqirtavjvu0t22ezrgzyorwxhpf3,Ajkal Khan,NaN,"my sister just received over 6,500 new <a rel=""nofollow"" class=""ot-hashtag"" href=""https://plus.google.com/s/%23active"">#active</a> youtube views Right now. The only thing she used was pimpmyviews....",1,eminem,1140,"my sister just received over 6,500 new <a rel=""nofollow"" class=""ot-hashtag"" href=""https://plus.google.com/s/%23active"">#active</a> youtube views Right now. The only thing she used was pimpmyviews...."
1142,z13xjfr42z3uxdz2223gx5rrzs3dt5hna,Jihad Naser,NaN,Hello I&#39;am from Palastine﻿,1,eminem,1142,Hello I&#39;am from Palastine﻿
1144,z12zgrw5furdsn0sc233hfwavnznyhicq,kyeman13,NaN,Go check out my rapping video called Four Wheels please ❤️﻿,1,eminem,1144,Go check out my rapping video called Four Wheels please ❤️﻿
1146,z12gxdortqzwhhqas04cfjrwituzghb5tvk0k,Muhammad Asim Mansha,NaN,Aslamu Lykum... From Pakistan﻿,1,eminem,1146,Aslamu Lykum... From Pakistan﻿
...,...,...,...,...,...,...,...,...
1578,LneaDw26bFv4NFg1m91sL1Aq1p-2N06VweQT4vOwTqc,Ando Nesia - | MC | Music Producer,NaN,DO YOU KNOW HOW SEAN KINGSTON GOT FAMOUS WHY DON&#39;T YOU LOOK IT UP KID BEFORE YOUR SO HARD ON YOURSELF!! IF YOU HIT ME UP WITH A MESSAGE IN MY INBOX AND SUBSCRIBE I WILL CHECK OUT YOUR CHANNEL...,1,eminem,1578,DO YOU KNOW HOW SEAN KINGSTON GOT FAMOUS WHY DON&#39;T YOU LOOK IT UP KID BEFORE YOUR SO HARD ON YOURSELF!! IF YOU HIT ME UP WITH A MESSAGE IN MY INBOX AND SUBSCRIBE I WILL CHECK OUT YOUR CHANNEL...
1580,LneaDw26bFsVQRSG0ecctIfRIn4Od1tJqvnhDwCNqbI,the34104,NaN,check out eminem latest track survival if u didnt,1,eminem,1580,check out eminem latest track survival if u didnt
1581,LneaDw26bFu3RCmyrWyP9S6wh1h9dBv3X95g1HzKAb4,Dany PK,NaN,SUBSCRIBE TO MY CHANNEL X PLEASE!. SPARE,1,eminem,1581,SUBSCRIBE TO MY CHANNEL X PLEASE!. SPARE
1582,LneaDw26bFsD65dtIvAEObWYIYnFTqQDKBek_Ypz3J8,SmexyFriedChicken,NaN,Check out my videos guy! :) Hope you guys had a good laugh :D,1,eminem,1582,Check out my videos guy! :) Hope you guys had a good laugh :D


label,1
source,
eminem,245


In [7]:
print("Размер:", df.shape)
display(df.head())

print("Состав по источникам и классам:")
display(
    pd.crosstab(df["source"], df["label"], margins=True)
    .reindex(list(files) + ["All"])
)

print("Пропуски по столбцам:")
display(df.isna().sum().to_frame("missing"))

empty_text = (
    df["text"].notna()
    & df["text"].str.replace(r"[\s\u200b\ufeff]", "", regex=True).eq("")
)
print("Пропущенных текстов:", df["text"].isna().sum())
print("Пустых текстов:", empty_text.sum())

missing_rows = df.isna().any(axis=1)
lost = df.loc[missing_rows]

print(f"При dropna() потеряем {len(lost)} строк ({missing_rows.mean():.1%}).")
print("Удаляемые строки по источникам и классам:")
display(pd.crosstab(lost["source"], lost["label"]))

print("Какие поля отсутствуют в удаляемых строках:")
display(lost.isna().sum().to_frame("missing"))

print("Примеры удаляемых строк:")
display(lost[["id", "source", "DATE", "text", "label"]].head(10))

Размер: (1956, 8)


,COMMENT_ID,AUTHOR,DATE,text,label,source,id,text_raw
0,LZQPQhLyRh80UYxNuaDWhIGQYNQ96IuCg-AYWqNPjpU,Julius NM,2013-11-07T06:20:48,"Huh, anyway check out this you[tube] channel: kobyoshi02",1,psy,0,"Huh, anyway check out this you[tube] channel: kobyoshi02"
1,LZQPQhLyRh_C2cTtd9MvFRJedxydaVW-2sNg5Diuo4A,adam riyati,2013-11-07T12:37:15,"Hey guys check out my new channel and our first vid THIS IS US THE MONKEYS!!! I'm the monkey in the white shirt,please leave a like comment and please subscribe!!!!",1,psy,1,"Hey guys check out my new channel and our first vid THIS IS US THE MONKEYS!!! I'm the monkey in the white shirt,please leave a like comment and please subscribe!!!!"
2,LZQPQhLyRh9MSZYnf8djyk0gEF9BHDPYrrK-qCczIY8,Evgeny Murashkin,2013-11-08T17:34:21,just for test I have to say murdev.com,1,psy,2,just for test I have to say murdev.com
3,z13jhp0bxqncu512g22wvzkasxmvvzjaz04,ElNino Melendez,2013-11-09T08:28:43,me shaking my sexy ass on my channel enjoy ^_^ ﻿,1,psy,3,me shaking my sexy ass on my channel enjoy ^_^ ﻿
4,z13fwbwp1oujthgqj04chlngpvzmtt3r3dw,GsMega,2013-11-10T16:05:38,watch?v=vtaRGgvGtWQ Check this out .﻿,1,psy,4,watch?v=vtaRGgvGtWQ Check this out .﻿


Состав по источникам и классам:


label,0,1,All
source,,,
psy,175,175,350
katyperry,175,175,350
lmfao,202,236,438
eminem,203,245,448
shakira,196,174,370
All,951,1005,1956


Пропуски по столбцам:


,missing
COMMENT_ID,0
AUTHOR,0
DATE,245
text,0
label,0
source,0
id,0
text_raw,0


... (2 outputs left out)

Какие поля отсутствуют в удаляемых строках:


,missing
COMMENT_ID,0
AUTHOR,0
DATE,245
text,0
label,0
source,0
id,0
text_raw,0


Примеры удаляемых строк:


,id,source,DATE,text,label
1138,1138,eminem,NaN,+447935454150 lovely girl talk to me xxx﻿,1
1140,1140,eminem,NaN,"my sister just received over 6,500 new <a rel=""nofollow"" class=""ot-hashtag"" href=""https://plus.google.com/s/%23active"">#active</a> youtube views Right now. The only thing she used was pimpmyviews....",1
1142,1142,eminem,NaN,Hello I&#39;am from Palastine﻿,1
1144,1144,eminem,NaN,Go check out my rapping video called Four Wheels please ❤️﻿,1
1146,1146,eminem,NaN,Aslamu Lykum... From Pakistan﻿,1
1148,1148,eminem,NaN,Help me get 50 subs please ﻿,1
1150,1150,eminem,NaN,"Alright ladies, if you like this song, then check out John Rage. He&#39;s a smoking hot rapper coming into the game. He&#39;s not better than Eminem lyrically, but he&#39;s hotter. Hear some of ...",1
1153,1153,eminem,NaN,"<a href=""https://www.facebook.com/groups/100877300245414/"">https://www.facebook.com/groups/100877300245414/</a>﻿",1
1154,1154,eminem,NaN,Take a look at this video on YouTube:﻿,1
1155,1155,eminem,NaN,Check out our Channel for nice Beats!!﻿,1


**Ответ:** Отсутсвует только дата. Для нашей модели это неважно, а строчки потярем. Так что удалять нет смыслы

### 1.2. Повторы и разбиение

Найдите точные повторы и проверьте - не скрываются ли одинаковые сообщения за разным оформлением? Выберите правила сравнения и создайте отдельный ключ, не меняя при этом исходный `text`. Обоснуйте свой выбор на нескольких группах из трейна.

> Для невидимых различий может пригодиться `repr()`

Решите, что делать с пропусками, пустыми текстами, повторами и возможными конфликтами меток. Объясните, какие строки сохраняете или исключаете.

Подготовьте трейн, валидацию и тест по заданному разбиению видео. В таблицах должны остаться `id`, `source`, `text`, `label` и `split`, порядок сохраняем по `id`.

Проверьте, что ID уникальны, в каждой части есть оба класса, а видео и одинаковые тексты не пересекаются между частями. Для текстов проверьте и точное совпадение, и ваш ключ. Если группа попала в разные части, выберите правило исключения записей. Повторы внутри части можно оставить с обоснованием.

Покажите размеры и состав классов до и после подготовки. Единого правильного числа строк нет. Зафиксируйте выборку: все следующие варианты модели сравниваем на одних и тех же строках, без удаления примеров по результатам предсказаний.

In [8]:
def text_key(text):
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = unicodedata.normalize("NFKC", text).casefold()
    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text)
    return re.sub(r"\s+", " ", text).strip()


data = df.copy()
data["split"] = data.source.map({
    "psy": "train", "katyperry": "train", "lmfao": "train",
    "eminem": "val", "shakira": "test"
})
data["key"] = data.text.fillna("").map(text_key)

display(data[data.duplicated("text", keep=False)]
        .sort_values(["text", "id"])[["id", "source", "text", "label"]])

# Смотрим разные оформления одного сообщения только на трейне
tr = data[data.split.eq("train")]
variants = tr.groupby("key").text.nunique()
keys = variants[variants > 1].index[:3]

for key in keys:
    display(
        tr.loc[tr.key.eq(key), ["id", "text", "label"]]
        .assign(text=lambda x: x.text.map(repr))
    )

# Убираем htmp и различия в регистре, пробелах и невидимых символах
# Пунктуацию, числа и адреса сохраняем: они могут быть признаками спама
before = pd.crosstab(data.split, data.label).reindex(["train", "val", "test"])

# Дата и автор для модели не нужны. Удаляем только строки без текста или метки
data = data.dropna(subset=["text", "label"])
data = data[data.key.ne("")].copy()

conflicts = data.groupby("key").label.nunique()
conflict_keys = conflicts[conflicts > 1].index
display(data[data.key.isin(conflict_keys)]
        .sort_values(["key", "id"])[["id", "source", "text", "label"]])

# Исключаем группы с противоречивыми метками целиком
data = data[~data.key.isin(conflict_keys)].copy()

# При пересечении сохраняем группу в одной части: train, затем val, затем test
priority = {"train": 0, "val": 1, "test": 2}
data["priority"] = data.split.map(priority)
first_split = data.groupby("key").priority.transform("min")
data = data[data.priority.eq(first_split)]

# Внутри части оставляем один пример на ключ, чтобы повторы не имели лишнего веса
data = data.sort_values("id").drop_duplicates(["split", "key"])

assert data.id.is_unique
assert data.groupby("split").label.apply(lambda x: set(x) == {0, 1}).all()

for column in ["source", "text", "key"]:
    assert data.groupby(column).split.nunique().le(1).all()

after = pd.crosstab(data.split, data.label).reindex(["train", "val", "test"])
sizes = pd.concat({"before": before, "after": after}, axis=1)
sizes[("before", "total")] = before.sum(axis=1)
sizes[("after", "total")] = after.sum(axis=1)
display(sizes)

columns = ["id", "source", "text", "label", "split"]
train = data.loc[data.split.eq("train"), columns].copy()
val = data.loc[data.split.eq("val"), columns].copy()
test = data.loc[data.split.eq("test"), columns].copy()

,id,source,text,label
1248,1248,eminem,Check out my SEXY VIDEO :*,1
1249,1249,eminem,Check out my SEXY VIDEO :*,1
1245,1245,eminem,subscribe to my feed,1
1763,1763,shakira,subscribe to my feed,1
1910,1910,shakira,******* Facebook is LAME and so 2004! Check out ------------ swagFriends com Make thousands of cool new friends everyday! Join this new movement!,1
...,...,...,...,...
1702,1702,shakira,waka waka﻿,0
1706,1706,shakira,wow,0
1822,1822,shakira,wow,0
1934,1934,shakira,wow,0


,id,text,label
860,860,'Awesome\ufeff',0
890,890,'awesome\ufeff',0
1125,1125,'awesome\ufeff',0


,id,text,label
579,579,'Best song ever \ufeff',0
880,880,'Best song ever\ufeff',0


,id,text,label
907,907,'Check out this playlist on YouTube:\ufeff',1
927,927,'Check out this playlist on YouTube:\ufeff',1
938,938,'Check out this playlist on YouTube:<br /><br />\ufeff',1
959,959,'Check out this playlist on YouTube:\ufeff',1
964,964,'Check out this playlist on YouTube:<br /><br />\ufeff',1
973,973,'Check out this playlist on YouTube:\ufeff',1
977,977,'Check out this playlist on YouTube:\ufeff',1
981,981,'Check out this playlist on YouTube:\ufeff',1
990,990,'Check out this playlist on YouTube:\ufeff',1
997,997,'Check out this playlist on YouTube:\ufeff',1


,id,source,text,label


before      after      before after
label      0    1     0    1  total total
split                                    
train    552  586   537  483   1138  1020
val      203  245   191  205    448   396
test     196  174   157  138    370   295

**Ответ:** Для поиска повторов убрал htmp, различия в регистре, пробелах и невидимых символах. Пунктуацию, числа и ссылки сохранил, исходный текст не менял.

Строки без текста или метки, группы с конфликтующими метками исключил. Пропуски в дате оставил, т.к. для классификации по тексту дата не нужна. Среди повторов внутри части сохранил один пример, а при пересечении частей использовал приоритет train -> val -> test, чтобы избежать утечки.

Разбиение по видео сохранил. Проверки уникальности id, наличия обоих классов и отсутствия пересечений прошли.

### 1.3. Что считать спамом

Сформулируйте своё правило: какие комментарии вы бы скрывали как спам?

Выберите **20 случайных комментариев из train** и поставьте свои метки: 0, 1 или «сомневаюсь», не глядя на метки корпуса. Затем сравните ответы и сохраните таблицу.

Давайте обсудим расхождения/пограничные случаи. Предложите конкретное уточнение инструкции разметчикам. Здесь оценивается именно обоснование. Но в обучении продолжаем использовать исходные метки.

**Баллы за задание 1:** 1 за подготовку и разбор проблем, 0.5 за проверки разбиения, 0.5 за собственную разметку и уточнение инструкции.

**Правило спама до просмотра исходных меток:**

**Ответ:** # your text here (ಠ.ಠ)

In [ ]:
# your code here
# (づ｡◕‿‿◕｡)づ
# Выбор комментариев и собственные метки.

In [ ]:
# your code here
# (⌐■_■)
# Сравнение с исходными метками.

**Сравнение меток и уточнение инструкции:**

**Ответ:** # your text here (ಠ.ಠ)

## Стартовая точка

Данные готовы. Начнём со знакомого словного TF-IDF и логистической регрессии.

В `make_model` можно передать `preprocessor` - вашу функцию «строка -> строка», и `feature_function` - функцию, которая получает исходные тексты и возвращает матрицу «число текстов × число признаков». `FeatureUnion` объединяет представления, масштабирование числовых признаков выполняется внутри `Pipeline`.

> Собственный `preprocessor` заменяет стандартную предобработку, включая перевод в нижний регистр. Решение о регистре нужно учесть в своей функции.

In [ ]:
def make_model(preprocessor=None,feature_function=None,use_words=True,C=1):
    branches=[]
    if use_words:
        branches.append(('words',TfidfVectorizer(
            min_df=2,max_features=10000,preprocessor=preprocessor,
            lowercase=preprocessor is None,
        )))
    if feature_function is not None:
        branches.append(('signals',Pipeline([
            ('extract',FunctionTransformer(feature_function,validate=False)),
            ('scale',StandardScaler()),
        ])))
    if not branches:
        raise ValueError('Нужен хотя бы один набор признаков')
    return Pipeline([
        ('features',FeatureUnion(branches)),
        ('classifier',LogisticRegression(C=C,max_iter=1000,random_state=13)),
    ])

In [ ]:
def quality(y_true, y_pred):
    return {
        'F1 spam': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision spam': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall spam': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

baseline = make_model()
baseline.fit(train['text'], train['label'])
baseline_prediction = baseline.predict(valid['text'])
display(pd.DataFrame([quality(valid['label'], baseline_prediction)], index=['TF-IDF']))

## 2. Обработка под задачу [2 балла]

Изучите train и выберите обработку, которая имеет смысл для этих комментариев. Реализуйте функцию «строка -> строка». Набор операций и инструменты выбираете сами.

Покажите действие функции на нескольких настоящих комментариях с разными особенностями. Что изменилось, какая информация сохранилась, а какая потерялась? Есть ли риск убрать полезный для классификации сигнал?

Проверьте повторное применение функции и появление пустых результатов. Объясните обнаруженные проблемы, если они есть. Строки из подготовленных выборок не удаляем.

**Баллы:** 1 за обоснование и реализацию обработки, 0.5 за примеры и анализ изменений, 0.5 за проверки.

In [ ]:
# your code here
# ٩(⁎❛ᴗ❛⁎)۶

In [ ]:
# your code here
# ヾ(๑╹◡╹)ﾉ

**Ответ:** # your text here (ಠ.ಠ)

## 3. Собственные признаки [2 балла]

Придумайте числовые или бинарные признаки, которые могут помочь отличить спам от обычных комментариев на YouTube. Каждый признак должен вычисляться по тексту комментария, по одному и тому же правилу для всех сообщений.

Для каждого признака объясните, что он измеряет и почему вы ожидаете, что он будет полезен. Набор и количество признаков выбирайте самостоятельно.

-> Реализуйте функцию, возвращающую матрицу признаков. Проверьте её размер и отсутствие пропусков или бесконечностей.

На трейне сравните распределения или сводную статистику признаков по классам. Подтвердились ли ваши предположения? Найдите настоящий комментарий, показывающий ограничение хотя бы одного признака.

**Баллы:** 1 за признаки с обоснованием, 0.5 за реализацию и проверки, 0.5 за статистику и контрпример.

In [ ]:
# your code here
# ฅ^•ﻌ•^ฅ

In [ ]:
# your code here
# ʕ•ᴥ•ʔ

**Ответ:** # your text here (ಠ.ಠ)

## 4. Что дает каждый набор признаков [1.5 балла]

### 4.1. Общий результат

Сравните на валидации четыре варианта:

| Вариант | Вход классификатора |
|---|---|
| Исходный | Готовый TF-IDF без вашей обработки |
| Обработка | TF-IDF после вашей функции |
| Собственные признаки | Только признаки из задания 3 |
| Вместе | TF-IDF после обработки + признаки из задания 3 |

Используйте одинаковые настройки логистической регрессии, включая `C`, и одинаковые параметры TF-IDF. Соберите таблицу с F1, precision и recall спама.

Какой вклад дали обработка и дополнительные признаки? Разберите два настоящих комментария из валидации, которые помогают объяснить сравнение моделей. Улучшение метрики при этом показывать не обязательно.

In [ ]:
# your code here
# (っ˘ڡ˘ς)

**Общий результат и разбор двух комментариев**

**Ответ:** # your text here (ಠ.ಠ)

### 4.2. Отдельный тип комментариев

По наблюдениям на трейне выберите один тип комментариев и задайте правило отбора, не зависящее от прогнозов модели. На этом срезе валидации сравните исходную модель с одной из доработанных: покажите метрики и число объектов каждого класса.

*Сохраняется ли вывод по общей метрике?*

Выберите кандидата для финальной проверки с учётом общего результата и среза. Объясните решение.

**Баллы за задание 4:** 0.5 за корректное сравнение четырёх вариантов, 0.5 за проверку и интерпретацию среза, 0.5 за разбор примеров и обоснованный выбор.

In [ ]:
# your code here
# (•̀ᴗ•́)و

**Результат на срезе и выбор модели:**

**Ответ:** # your text here (ಠ.ಠ)

## 5. Проверки на измененных текстах [1.5 балла]

Выберите **8 настоящих комментариев из валидации**, лучше оба класса. Придумайте **два преобразования**, которые сохраняют смысл и метку этих сообщений.

А получится хотя бы одно преобразование связать с риском вашей обработки/признаков? Объясните, что может измениться в представлении текста, доступном модели.

Без переобучения сравните исходную модель с одной из доработанных. Для каждой модели и преобразования посчитайте новые и исправленные ошибки.

Разберите какой-нибудь показательный пример. Если новых ошибок нет, объясните, чего эта проверка всё ещё не гарантирует?

**Баллы:** 0.5 за обоснование преобразований и сохранения метки, 0.5 за таблицу и подсчёты, 0.5 за разбор результата и связь с представлением текста.

In [ ]:
# your code here
# =^･ｪ･^=

**Ответ:** # your text here (ಠ.ಠ)

## 6. Новое видео [1 балл]

Зафиксируйте окончательный выбор до оценки на тесте - это наша последняя граница. Обучите весь выбранный `Pipeline` заново на train + validation, включая словарь, IDF и масштабирование, если они используются. Оцените его на тесте: F1, precision, recall спама и матрица ошибок.

Сохраните `test_predictions.csv` с колонками `id` и `prediction` для всех строк вашего test в порядке `id`. Мы этот файл будем прикреплять вместе с ноутбуком в lms.

Переносится ли решение на новое видео? Какие ошибки остаются и что нужно проверить перед реальным использованием?

**Баллы:** 0.5 за итоговую оценку и файл, 0.5 за обоснованный вывод и воспроизводимость.

In [ ]:
# your code here
# (⊙_⊙)

**Ответ:** # your text here (ಠ.ಠ)

## Источники и помощь

Укажите использованные материалы и помощь генеративных моделей. Если заимствовали код, отметьте источник и объясните, что изменили.

**Ответ:** # your text here (ಠ.ಠ)